# Submission TS2-B: soft voting with team features (species pairs), all features

This notebook runs end to end from the raw competition files (`train.jsonl`, `test.jsonl`, `type_chart_gen2.csv`,
`sample_submission.csv`): preprocessing, move-damage table, species table, features, model and submission. It reads no
intermediate files (`data/processed/`, saved models or `best_params.json`); the tuned parameters are written in the notebook.

The project code comes from our repository ([mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version](https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version)), which is cloned
automatically when the notebook is not run from inside the repo (e.g. on Kaggle, where internet access must be on).

Sections: 1. setup, 2. data, 3. preprocessing, 4. features, 5. model, 6. validation (train only), 7. final fit and submission.

## 1. Setup

In [1]:
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/mogliaaaa/FDS-Pokemon-Battles-Prediction-2026-Team-Plasma-Version.git"
REPO_REF = "main"   # pin a commit hash or tag for the final, exactly reproducible run

def find_repo_root():
    # Local run: the notebook is in the repo (root or notebooks/)
    for p in (Path.cwd(), Path.cwd().parent):
        if (p / "src" / "features.py").exists():
            return p.resolve()
    # Kaggle (or anywhere else): clone the repository next to the notebook
    target = Path("PokePredict")
    if not (target / "src" / "features.py").exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", REPO_REF], check=True)
    return target.resolve()

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))
commit = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"repo: {ROOT} | commit: {commit or 'unknown'}")

from sklearn.base import clone
from sklearn.ensemble import VotingClassifier

from src import preprocessing as pp
from src.data import find_data_dir, load_jsonl, load_type_chart
from src.features import build_features, fit_team_cooccurrence, robust_columns
from src.models import (MONOTONE_DIRECTIONS, get_model_specs, monotone_params, params_to_text,
                        seed_bagged, with_params)
from src.move_damage import fit_move_damage
from src.team_strength import unseen_cv_scores, with_team_strength
from src.validation import SEED, cv_scores

RUN_CV = True   # 5-fold CV on train before the final fit (set to False to only write the submission)

repo: .. | commit: d6c7ae9


## 2. Data

On Kaggle the files are in `/kaggle/input/...`, locally in `data/raw/`; `find_data_dir` checks both.

In [2]:
DATA_DIR = find_data_dir()
train = load_jsonl(DATA_DIR / "train.jsonl")
test = load_jsonl(DATA_DIR / "test.jsonl")
type_chart = load_type_chart(DATA_DIR / "type_chart_gen2.csv")
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
print(f"data dir: {DATA_DIR} | train: {len(train):,} | test: {len(test):,} | sample submission: {len(sample):,}")

data dir: ..\data\raw | train: 10,000 | test: 5,000 | sample submission: 5,000


## 3. Preprocessing

Same steps as `notebooks/02_preproc.ipynb` (code in `src/preprocessing.py` and `src/move_damage.py`):
- `team_frame`: P1's team and P2's lead, with normalised types and an under-level flag;
- `fit_move_damage`: expected damage of each move, estimated from the observed HP drops, so that moves with variable
  power (base power 0 in the data) get a realistic value. Fitted on train only;
- `timeline_frame`: one row per battle, side and turn, with the action labels and the expected damage;
- `pokemon_states`: last known state of every Pokémon seen in the 30 turns (KO, HP, vanished);
- `build_species_table`: base stats and types per species, computed on train only (species missing from train get the
  mean).

In [3]:
t0 = time.time()
damage_table = fit_move_damage(train)
species = pp.build_species_table(train)

def preprocess(battles):
    team = pp.team_frame(battles)
    timeline = pp.timeline_frame(battles, damage_table)
    states = pp.pokemon_states(timeline)
    return team, timeline, states

tables_train = preprocess(train)
tables_test = preprocess(test)
print(f"preprocessing done in {time.time() - t0:.0f}s")

preprocessing done in 7s


## 4. Features (v1 + team keys)

Same as `notebooks/03_features.ipynb` (`src/features.py`): 155 numeric features in 7 groups, plus two team keys:
`key_p1_team` (P1's six species) and `key_p2_seen` (the P2 species revealed in the 30 turns). The keys are text, so they
are not passed to the models directly: the first step of the pipeline turns them into the team-strength features
(section 7), fitted on the training data only.

In [4]:
X_train, groups = build_features(*tables_train, species, type_chart, team_keys=True)
X_test, _ = build_features(*tables_test, species, type_chart, team_keys=True)
y = X_train.pop("player_won")

CAT = groups["species_cat"]
KEYS = groups.get("team_keys", [])     # text keys, encoded inside the pipeline (team-strength variants only)
NUM = [c for c in X_train.columns if c not in CAT + KEYS]
ROBUST = robust_columns(NUM)

FEATURES = NUM + KEYS

X_train, X_test = X_train[FEATURES], X_test[FEATURES]
assert list(X_train.columns) == list(X_test.columns)
print(f"train {X_train.shape} | test {X_test.shape} | features used: {len(FEATURES)}")
if len(FEATURES) < len(NUM):
    print("removed (fragile, based on P2's recorded KOs):", sorted(set(NUM) - set(FEATURES)))

train (10000, 157) | test (5000, 157) | features used: 157


## 5. Model

Parameters tuned in experiment 04. Why we chose this model is explained in section 7.

In [5]:
# Tuned in notebooks/04_models.ipynb (experiment 04, RandomizedSearchCV, 5-fold CV, seed 42)
PARAMS = {
    'xgboost': {
        'model__colsample_bytree': 0.6856273981493477,
        'model__gamma': 0.010493314645539311,
        'model__learning_rate': 0.012145928554953287,
        'model__max_depth': 6,
        'model__min_child_weight': 1.3386261584543908,
        'model__n_estimators': 720,
        'model__reg_lambda': 3.5416296712651336,
        'model__subsample': 0.5993578407670862,
    },
    'lightgbm': {
        'model__colsample_bytree': 0.6286197472598067,
        'model__learning_rate': 0.006552992991573526,
        'model__min_child_samples': 106,
        'model__n_estimators': 1424,
        'model__num_leaves': 12,
        'model__reg_alpha': 0.6070874110546393,
        'model__reg_lambda': 2.7129089558470034,
        'model__subsample': 0.6629794526009424,
    },
    'catboost': {
        'model__bagging_temperature': 0.15071754396542947,
        'model__depth': 5,
        'model__iterations': 763,
        'model__l2_leaf_reg': 1.14876138417489,
        'model__learning_rate': 0.03898458749305194,
        'model__random_strength': 1.3551287236845648,
    },
    'gradient_boosting': {
        'model__learning_rate': 0.04760153910846178,
        'model__max_depth': 3,
        'model__max_features': None,
        'model__min_samples_leaf': 91,
        'model__n_estimators': 466,
        'model__subsample': 0.9314950036607718,
    },
}

specs = get_model_specs()
for n in PARAMS:
    print(f"{n:18s} {params_to_text(PARAMS[n])}")

ENCODER_PARAMS = {'train_unseen': 0.3, 'pairs': True, 'per_side': True}   # team encoder, R1 + R3 of notebook 26
voting = VotingClassifier([(n, with_params(specs[n], PARAMS[n])) for n in PARAMS], voting="soft", n_jobs=1)
model = with_team_strength(voting, **ENCODER_PARAMS)
baseline = clone(voting)
model

xgboost            colsample_bytree=0.6856, gamma=0.01049, learning_rate=0.01215, max_depth=6, min_child_weight=1.339, n_estimators=720, reg_lambda=3.542, subsample=0.5994
lightgbm           colsample_bytree=0.6286, learning_rate=0.006553, min_child_samples=106, n_estimators=1424, num_leaves=12, reg_alpha=0.6071, reg_lambda=2.713, subsample=0.663
catboost           bagging_temperature=0.1507, depth=5, iterations=763, l2_leaf_reg=1.149, learning_rate=0.03898, random_strength=1.355
gradient_boosting  learning_rate=0.0476, max_depth=3, max_features=None, min_samples_leaf=91, n_estimators=466, subsample=0.9315


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('team', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,train_unseen,0.3
,pairs,True
,per_side,True
,smoothing,10.0
,pooled,True
,n_inner,5
,counts,True


## 6. Validation (train only)

Stratified 5-fold CV with `shuffle=True` and seed 42, as in all our experiments. Every step that learns from the data is
part of the model, so it is refitted on each training fold, and the test set is never used. The reference values of
experiment 04 in section 7 come from repeated CV over 3 seeds, so a single seed can differ by a few tenths of a point.

In [6]:
if RUN_CV:
    t0 = time.time()
    scores = cv_scores(model, X_train, y)
    print("accuracy per fold:", scores.round(4))
    print(f"5-fold CV: {scores.mean():.4f} ± {scores.std():.4f}  ({time.time() - t0:.0f}s)")
    # Paired comparison: the experiment-04 model on the same folds
    base = cv_scores(baseline, X_train.drop(columns=KEYS), y)
    gain = scores - base
    print("baseline per fold:", base.round(4))
    print(f"baseline (experiment-04 model): {base.mean():.4f} | gain {gain.mean() * 100:+.2f} points, "
          f"better in {(gain > 0).sum()}/5 folds")
    # Teams never seen: the validation teams are removed from the encoder's pool (notebook 26)
    for side in ("p1", "both"):
        unseen = unseen_cv_scores(model, X_train, y, side=side)
        print(f"unseen teams ({side}): {unseen.mean():.4f} | vs experiment-04 model {(unseen - base).mean() * 100:+.2f} points, "
              f"better in {(unseen > base).sum()}/5 folds")
else:
    print("CV skipped (RUN_CV = False)")

accuracy per fold: [0.693  0.6885 0.6905 0.69   0.6835]
5-fold CV: 0.6891 ± 0.0032  (64s)
baseline per fold: [0.6745 0.683  0.691  0.6825 0.675 ]
baseline (experiment-04 model): 0.6812 | gain +0.79 points, better in 4/5 folds
unseen teams (p1): 0.6830 | vs experiment-04 model +0.18 points, better in 3/5 folds
unseen teams (both): 0.6795 | vs experiment-04 model -0.17 points, better in 2/5 folds


## 7. Final fit and submission

**Approach.** The soft-voting ensemble of submission 1 (XGBoost, LightGBM, CatBoost and Gradient Boosting with the
parameters tuned in experiment 04), trained on the 155 v1 features plus the team features with species pairs. Notebook 26
only measured LightGBM, so section 6 gives the numbers for the ensemble.

**Team features (experiments 11 and 13).** Teams repeat across battles: there are about 4,200 distinct P1 teams in 10,000
battles, and P2's teams come from the same pool. `TeamStrengthEncoder` (`src/team_strength.py`) is the first step of the
pipeline, so it is refitted on each training fold and no battle ever sees its own label. It computes the win rate of P1's
exact team and of the pool teams compatible with the P2 Pokémon revealed so far, how many battles each rate is based on
and, since experiment 13, species pair scores (`pairs=True`), which still give a value for a new team made of known
species. During training the team win rates are hidden at random, separately for each side (`train_unseen=0.3`,
`per_side=True`). A large part of the signal is rarity: a P1 team that appears only once wins about 40% of its battles,
and we don't know why.

In experiment 13 (notebook 26, LightGBM with all features, 15 folds), compared with the encoder of submissions 22 and 24,
this setup gained +0.21 points on the normal CV (8/15 folds). With the validation teams removed from the pool it gained
+0.43 (P1 only, 15/15 folds) and +0.287 (both sides, 12/15). Our rule, fixed before running, required +0.3 on both, so
this configuration missed it by 0.013 points. We used it anyway, because it was not worse in any scenario and it is
close to neutral on new teams (+0.09 and -0.08 against the model without team features). Section 6 also scores the
experiment-04 model on the same folds and measures this model on teams never seen. New species are not covered.

The model is trained on the full train set (10,000 battles) and predicts the test battles. The output follows
`sample_submission.csv` (`battle_id,player_won`): on Kaggle it is written to `/kaggle/working/submission.csv`, locally
to `submissions/submission_ts2_main.csv`.

In [7]:
t0 = time.time()
model.fit(X_train, y)
pred = pd.Series(model.predict(X_test).astype(int), index=X_test.index)
print(f"fitted and predicted in {time.time() - t0:.0f}s")

submission = sample[["battle_id"]].copy()
submission["player_won"] = submission["battle_id"].map(pred)
assert submission["player_won"].notna().all(), "some test battles have no prediction"
submission["player_won"] = submission["player_won"].astype(int)

kaggle_dir = Path("/kaggle/working")
out_path = kaggle_dir / "submission.csv" if kaggle_dir.exists() else ROOT / "submissions" / "submission_ts2_main.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)
print(f"saved: {out_path} | rows: {len(submission):,}")
print("predicted P1 win rate:", round(submission["player_won"].mean(), 4))
submission.head()

fitted and predicted in 64s
saved: ..\submissions\submission_ts2_main.csv | rows: 5,000
predicted P1 win rate: 0.5152


,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1
